In [ ]:
"""
Concepts:

1. Gini
2. Entropy
3. Information Gain
4. Tree Visualization
5. Pruning
6. Feature Importance
7. Overfitting Detection
8. Decision Path Analysis
"""

# =====================================================
# IMPORTS
# =====================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split,cross_val_score,GridSearchCV,RandomizedSearchCV,learning_curve)
from sklearn.tree import (DecisionTreeClassifier,plot_tree)
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix,classification_report,roc_curve,precision_recall_curve)
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve

In [ ]:
# =====================================================
# STEP 1 : DATA UNDERSTANDING
# =====================================================

data = load_breast_cancer()
df = pd.DataFrame(data.data,columns=data.feature_names)
df["target"] = data.target
print(df.head())
print(df.info())
print(df.describe())
print(df["target"].value_counts())

In [ ]:
# =====================================================
# STEP 2 : EDA
# =====================================================

print(df.isnull().sum())
print(df.duplicated().sum())

In [ ]:
# =====================================================
# STEP 3 : TRAIN / VALIDATION / TEST
# =====================================================

X = df.drop("target", axis=1)
y = df["target"]

X_temp, X_test, y_temp, y_test = train_test_split(X,y,test_size=0.15,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp,y_temp,test_size=0.1765,random_state=42)

# =====================================================
# STEP 4 : PREPROCESSING
# =====================================================

# Trees do NOT require scaling


In [ ]:
# =====================================================
# STEP 5 : BASELINE MODEL
# =====================================================

dt = DecisionTreeClassifier(criterion="gini",random_state=42)
dt.fit(X_train,y_train)

In [ ]:
# =====================================================
# STEP 6 : VALIDATION METRICS with base model
# =====================================================

y_pred = dt.predict(X_val)
val_prob = dt.predict_proba(X_val)[:,1]

print("\nBASELINE MODEL")
accuracy_score(y_val,y_pred)
precision_score(y_val,y_pred)
recall_score(y_val,y_pred)
f1_score(y_val,y_pred)
roc_auc_score(y_val,y_pred)

In [ ]:
# =====================================================
# STEP 7 : CROSS VALIDATION
# =====================================================

cv_scores = cross_val_score(dt,X_train,y_train,cv=5,scoring="roc_auc",n_jobs=-1)
cv_scores.mean()
cv_scores.std()


In [ ]:
# =====================================================
# STEP 8 : HYPERPARAMETER TUNING
# =====================================================

param_grid = {
    "criterion":["gini","entropy"],
    "max_depth":[3,5,7,10,None],
    "min_samples_split":[2,5,10,20],
    "min_samples_leaf":[1,2,5,10],
    "ccp_alpha":[0.0,0.001,0.01]}

grid = GridSearchCV(DecisionTreeClassifier(random_state=42),param_grid,cv=5,scoring="roc_auc",n_jobs=-1)
grid.fit(X_train,y_train)
print(grid.best_params_)
best_model = grid.best_estimator_

In [ ]:
# =====================================================
# STEP 9 : VALIDATION with best model
# =====================================================

y_best = best_model.predict(X_val)
y_best_prob = best_model.predict_proba(X_val)[:,1]
val_auc = roc_auc_score(y_val,y_best_prob)

In [ ]:
# =====================================================
# STEP 10 : TRAIN VS VALIDATION
# =====================================================

train_prob = best_model.predict_proba(X_train)[:,1]
train_auc = roc_auc_score(y_train,train_prob)

In [ ]:
# =====================================================
# STEP 11 : TREE ANALYSIS
# =====================================================

# ----------------------------------------
# TREE DEPTH
# ----------------------------------------

print("\nTree Depth:",best_model.get_depth())
print("Leaf Nodes:",best_model.get_n_leaves())

# ----------------------------------------
# FEATURE IMPORTANCE
# ----------------------------------------

importance_df = pd.DataFrame({"Feature":X.columns,"Importance":best_model.feature_importances_}).sort_values(by="Importance",ascending=False)
print(importance_df.head(15))
plt.figure(figsize=(8,5))
plt.barh(importance_df["Feature"][:15],importance_df["Importance"][:15])
plt.title("Feature Importance")
plt.show()

# ----------------------------------------
# PERMUTATION IMPORTANCE
# ----------------------------------------

perm = permutation_importance(best_model,X_val,y_val,random_state=42)
perm_df = pd.DataFrame({"Feature":X.columns,"Importance":perm.importances_mean})

print("\nPermutation Importance")
print(perm_df.sort_values(by="Importance",ascending=False).head(15))

# ----------------------------------------
# CONFUSION MATRIX
# ----------------------------------------

cm = confusion_matrix(y_val,y_pred)
sns.heatmap(cm,annot=True,fmt='d')
plt.title("Confusion Matrix")
plt.show()

# ----------------------------------------
# ROC CURVE
# ----------------------------------------

fpr,tpr,_ = roc_curve(y_val,val_prob)
plt.plot(fpr,tpr)
plt.plot([0,1],[0,1])
plt.title("ROC Curve")
plt.show()

# ----------------------------------------
# PR CURVE
# ----------------------------------------

precision, recall, _ = precision_recall_curve(y_val,val_prob)
plt.plot(recall,precision)
plt.title("PR Curve")
plt.show()

# ----------------------------------------
# CALIBRATION CURVE
# ----------------------------------------

prob_true, prob_pred = calibration_curve(y_val,val_prob,n_bins=10)
plt.plot(prob_pred,prob_true,marker='o')
plt.plot([0,1],[0,1])
plt.title("Calibration Curve")
plt.show()

# ----------------------------------------
# MAX DEPTH ANALYSIS
# ----------------------------------------

depths = [2,3,4,5,7,10,15]

scores = []

for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth,random_state=42)
    model.fit(X_train,y_train)
    probs = model.predict_proba(X_val)[:,1]
    score = roc_auc_score(y_val,probs)
    scores.append(score)

plt.plot(depths,scores,marker='o')
plt.title("Depth Analysis")
plt.show()

# ----------------------------------------
# LEARNING CURVE
# ----------------------------------------

train_sizes, train_scores, val_scores = learning_curve(best_model,X_train,y_train,cv=5,scoring="roc_auc",n_jobs=-1)

plt.plot(train_sizes,np.mean(train_scores,axis=1),label="Train")
plt.plot(train_sizes,np.mean(val_scores,axis=1),label="Validation")
plt.legend()
plt.title("Learning Curve")
plt.show()

# ----------------------------------------
# TREE VISUALIZATION
# ----------------------------------------

plt.figure(figsize=(20,10))
plot_tree(best_model,feature_names=X.columns,class_names=["Malignant","Benign"],filled=True,max_depth=3)
plt.show()


In [ ]:
# =====================================================
# STEP 12 : FINAL MODEL
# =====================================================

final_model = best_model

In [ ]:
# =====================================================
# STEP 13 : TEST EVALUATION
# =====================================================

test_pred = final_model.predict(X_test)

test_prob = final_model.predict_proba(
    X_test
)[:,1]

print(classification_report(y_test,test_pred))
print("Test ROC-AUC:",roc_auc_score(y_test,test_prob))

In [ ]:
# =====================================================
# STEP 14 : DEPLOYMENT READINESS
# =====================================================

print("\nDeployment Metrics")
print("CV ROC-AUC:",cv_scores.mean())
print("Validation ROC-AUC:",val_auc)
print("Test ROC-AUC:",roc_auc_score(y_test,test_prob))

In [ ]:
# =====================================================
# STEP 15 : INTERVIEW QUESTIONS
# =====================================================

'''
1. What is Gini Impurity?
2. What is Entropy?
3. Difference between Gini and Entropy?
4. What is Information Gain?
5. How does a tree choose splits?
6. Why do trees overfit?
7. What is max_depth?
8. What is pruning?
9. What is ccp_alpha?
10. What is min_samples_split?
11. What is min_samples_leaf?
12. Why don't trees require scaling?
13. What is feature importance?
14. Feature Importance vs Permutation Importance?
15. Why are trees unstable?
16. What is a leaf node?
17. What is a root node?
18. What is a decision path?
19. Why Random Forest performs better?
20. Explain Decision Tree end-to-end.
'''